# SQLibrarian: evaluate a trained model (Kaggle, ~10 minutes)

Asks an already trained model (a merged model on the Hub, e.g. `OmarAmir2001/sqlibrarian-sft-run5`)
the dev questions twice: with the normal schema, and with the held-out renamed schema it never
saw in training. No training. Cost: free (Kaggle GPU quota, about 10 minutes).

**Before you run it:** Settings: GPU T4, Internet on. Secret `HF_TOKEN` (the model repo is private).
Then **Run All**, download `eval_predictions.zip`, and score both files on the laptop (section F).

| Section | What it does | Done when |
|---|---|---|
| A | Settings | printed |
| B | Secrets | names listed |
| C | Clone and install | "all imports OK" |
| D | GPU check | a Tesla T4 is listed |
| E | Generate (normal + held-out) | two `generate.done` lines |
| F | Pack | `eval_predictions.zip` in Output |

## A. Settings

In [ ]:
import os

# One GPU: a single process seeing both T4s makes Trainer use nn.DataParallel,
# which TRL refuses. A 1.5B LoRA run fits on one T4. Every later step inherits this.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

REPO_URL = "https://github.com/OmarAmir2001/sqlibrarian.git"
BRANCH = "master"
MODEL = "OmarAmir2001/sqlibrarian-sft-run5"  # merged model on the Hub
RUN_NAME = "sft-run5-eval"  # name of the prediction files
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/sqlibrarian-repo"
print(BRANCH, MODEL, RUN_NAME, sep="\n")


def run(cmd: str) -> None:
    """Run a shell command like `!cmd`, but STOP the notebook if it fails,
    so a failed step doesn't waste GPU time on the steps after it."""
    ip = get_ipython()
    ip.system(cmd)
    code = ip.user_ns.get("_exit_code", 0)
    if code != 0:
        raise RuntimeError(f"step failed (exit {code}): {cmd}")

## B. Secrets
Read from Kaggle secrets into environment variables, where `helpers/config.py` finds them.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in [
    "HF_TOKEN",
    "MLFLOW_TRACKING_URI",
    "MLFLOW_TRACKING_USERNAME",
    "MLFLOW_TRACKING_PASSWORD",
]:
    try:
        os.environ[name] = secrets.get_secret(name)
        print("set:", name)
    except Exception:
        print("MISSING:", name)

## C. Get the code

In [ ]:
import subprocess
import sys

os.chdir(WORK)  # step out of an old clone, so re-running is safe
subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True
)
%cd $REPO_DIR
!git log --oneline -1

In [ ]:
!pip install -q uv
!uv pip install --system -q --no-deps -e .
!uv pip install --system -q sqlglot structlog pydantic-settings pyyaml
# Kaggle ships an old torchao (0.10); new PEFT refuses to load next to it. We don't use it.
!pip uninstall -y -q torchao

In [ ]:
check = (
    "import sqlibrarian.evaluation.generate, transformers, torch; "
    "print('all imports OK | transformers', transformers.__version__, '| torch', torch.__version__)"
)
subprocess.run([sys.executable, "-c", check], check=True)

## D. GPU

In [ ]:
run("nvidia-smi --query-gpu=name,memory.total --format=csv")

## E. Generate: the normal schema, then the held-out one

In [ ]:
run(f"python -m sqlibrarian.evaluation.generate --model {MODEL} --run-name {RUN_NAME}")
run(
    f"python -m sqlibrarian.evaluation.generate --model {MODEL} --run-name {RUN_NAME}-heldout"
    " --schema data/eval/schema_heldout.txt --questions data/eval/dev_questions_heldout.jsonl"
)

## F. Pack the predictions
Download `eval_predictions.zip` from Output. On your laptop (the quotes matter if the browser adds " (2)" to the name):
```bash
unzip -o "notes/eval_predictions.zip" -d /tmp/preds && cp /tmp/preds/predictions/* data/eval/predictions/
uv run --with mlflow-skinny python -m sqlibrarian.evaluation.score data/eval/predictions/<run>.jsonl --mlflow
uv run --with mlflow-skinny python -m sqlibrarian.evaluation.score --variant heldout data/eval/predictions/<run>-heldout.jsonl --mlflow
```

In [ ]:
run("cd data/eval && zip -q -r /kaggle/working/eval_predictions.zip predictions")
print("saved /kaggle/working/eval_predictions.zip")